# 🚗 SIH 2024: Multi-Camera Vehicle ANPR & Re-ID Trajectory Tracking
### Problem Statement: 26127 (Bharat Electronics Limited)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23f2004399/SIH-ANPR-Tracking/blob/main/SIH_MultiCamera_ANPR_Colab.ipynb)

---

### 🎯 Overview & Key Features:
1. **High-Speed Vehicle Tracking**: YOLO11 (`yolo11n.pt`) with ByteTrack for multi-camera tracking.
2. **License Plate Localization**: Fine-tuned plate detection (`yolov8n_plate.pt`).
3. **Multi-Pass PaddleOCR**: Text detection (DBNet) + recognition with Indian state grammar and character confusion repair.
4. **State-of-the-Art Re-ID Embeddings**: Upgraded to **Meta DINOv2 (ViT-S/14)** with **Hybrid Color-Aware Palette Fusion** (0.85 Deep Features + 0.15 Central HSV Palette) for instant rejection of color mismatches.
5. **Bidirectional Trajectory Reconstruction**: Models incoming and outgoing vehicle movement across a 2-camera junction network.
6. **Visual Pop-Up Dashboard**: Side-by-side Top-5 candidate matching with telemetry and route cards rendered inline.

> 💡 **Hardware**: Run this notebook on a **T4 GPU** (Runtime → Change runtime type → T4 GPU).

## ⚙️ Step 1: Check GPU & Environment
Verify that an NVIDIA GPU is assigned to this session.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")

## 📦 Step 2: Clone Repository & Install Dependencies
Clone the SIH project repo and install the required vision packages (`ultralytics`, `paddlepaddle-gpu`, `paddleocr`).

In [ ]:
import os

# 1. Clone repository if not already present
if not os.path.isdir("SIH-ANPR-Tracking"):
    !git clone https://github.com/23f2004399/SIH-ANPR-Tracking.git
%cd SIH-ANPR-Tracking

# 2. Pull latest code
!git pull

# 3. Install core dependencies
!pip install --upgrade pip -q
!pip install -q ultralytics opencv-python pillow numpy pandas

# 4. Install PaddleOCR & PaddlePaddle GPU
!pip install -q paddlepaddle-gpu paddleocr>=2.7.3

print("\n✅ All dependencies installed successfully!")

## 📹 Step 3: Provide the 3 Camera Videos
Place your 3 input video files in the `input_videos/` directory.

Choose **one** of the methods below:
- **Option A (Google Drive - Recommended for large files)**: Mount Google Drive and copy files.
- **Option B (Direct Upload)**: Use the Colab file picker to upload.
- **Option C (Cloud URL / Wget)**: Download directly via URL.

In [ ]:
import os
os.makedirs("input_videos", exist_ok=True)

# =====================================================================
# OPTION A: Mount Google Drive (Uncomment to use)
# =====================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/CAM1.mp4 input_videos/cam1.mp4
# !cp /content/drive/MyDrive/CAM2.mp4 input_videos/cam2.mp4
# !cp /content/drive/MyDrive/CAM3.mp4 input_videos/cam3.mp4

# =====================================================================
# OPTION B: Direct File Upload (Uncomment to upload from your PC)
# =====================================================================
# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded.keys():
#     os.rename(fname, os.path.join("input_videos", fname))

# Verify files in input_videos
video_extensions = ('.mp4', '.avi', '.mkv', '.mov')
found_videos = sorted([
    os.path.join("input_videos", f) for f in os.listdir("input_videos")
    if f.endswith(video_extensions)
])

print(f"\n📁 Found {len(found_videos)} video(s) in input_videos/:")
for v in found_videos:
    size_mb = os.path.getsize(v) / (1024 * 1024)
    print(f"   • {v} ({size_mb:.1f} MB)")

if len(found_videos) < 3:
    print("\n⚠️ Note: You need 3 video files (Camera 1, Camera 2, Camera 3) for the full 3-camera network.")
    print("Please upload your videos into the 'input_videos/' folder or mount Google Drive.")

## 🧠 Step 4: Verify Models & Upgraded Re-ID Extractor
Test that `yolov8n_plate.pt` is present and test the **DINOv2 + Hybrid Color-Aware Extractor**.

In [ ]:
import os
assert os.path.exists("yolov8n_plate.pt"), "yolov8n_plate.pt missing from repository root!"
print("✅ License plate model found: yolov8n_plate.pt")

# Test Re-ID Extractor
from pipeline.reid import get_reid_model
import numpy as np

print("\n⚙️ Initializing Upgraded Re-ID Extractor [dinov2 + hybrid_color]...")
reid_extractor = get_reid_model(model_name="dinov2", device="cuda" if torch.cuda.is_available() else "cpu", hybrid_color=True)

# Test dummy inference
dummy_crop = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
emb = reid_extractor.get_embedding(dummy_crop)
print(f"✅ Extracted vector shape : {len(emb)}-dimensional normalized vector")
print(f"✅ Feature Composition   : 384-d Meta DINOv2 ViT-S + 16-d Normalized HSV Color Palette")
print(f"✅ Vector L2 norm        : {np.linalg.norm(emb):.4f} (Ready for cosine similarity dot product)")

## 🚀 Step 5: Run Multi-Camera ANPR & Re-ID Pipeline (`main.py`)
Processes the 3 videos sequentially:
- **Video 1** $\rightarrow$ `Camera_1` (corridor entry / main road)
- **Video 2** $\rightarrow$ `Camera_2` (fork branch A)
- **Video 3** $\rightarrow$ `Camera_3` (fork branch B)

Extracts high-resolution vehicle crops, runs PaddleOCR, generates DINOv2 hybrid embeddings, and writes telemetry to `vehicle_logs.csv`.

In [ ]:
import glob

# Discover videos in input_videos
all_input_videos = sorted(glob.glob("input_videos/*.*mp4"))
if not all_input_videos:
    all_input_videos = sorted(glob.glob("input_videos/*"))

assert len(all_input_videos) >= 1, "No video files found in input_videos/! Please add videos first."

target_videos = all_input_videos[:3] # Process up to 3 cameras
video_args_str = " ".join(f"'{v}'" for v in target_videos)

print(f"Processing {len(target_videos)} camera feed(s):\n{video_args_str}\n")

# Run pipeline with GPU acceleration & DINOv2 embeddings
!python pipeline/main.py \
    --videos {video_args_str} \
    --output_dir ./outputs_colab \
    --reid_model dinov2 \
    --hybrid_color \
    --device cuda \
    --conf_thresh 0.35 \
    --plate_conf_thresh 0.25 \
    --ocr_interval 2 \
    --save_all_crops

## 📊 Step 6: Inspect Telemetry & Consolidated Vehicle Logs (`vehicle_logs.csv`)
Check the logged vehicles, entry/exit timestamps, OCR plates, and embedding paths.

In [ ]:
import os
import pandas as pd

csv_path = "./outputs_colab/vehicle_logs.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=" * 75)
    print(f"📊 Total Vehicles Tracked Across Cameras: {len(df):,}")
    print("=" * 75)
    print("Sightings per Camera:")
    print(df['camera_id'].value_counts())
    print("-" * 75)
    
    confident_plates = df[df['plate_number'] != 'UNKNOWN']
    print(f"Confidently Read Number Plates: {len(confident_plates)} ({len(confident_plates)/max(1, len(df))*100:.1f}%)")
    if not confident_plates.empty:
        print(confident_plates[['camera_id', 'track_id', 'plate_number', 'average_ocr_confidence', 'entry_timestamp']].head(10))
    
    print("-" * 75)
    display(df.head(15))
else:
    print(f"❌ File not found: {csv_path}")

## 🔍 Step 7: Cross-Camera Re-ID & Trajectory Tracking (Random Sample)
Pick a random vehicle crop with `--one` and track its cross-camera trajectory.
Enforces the 2-camera corridor topology, models bidirectional traffic, and renders the visual dashboard inline.

In [ ]:
!python pipeline/test_embeddings.py \
    --output_dir ./outputs_colab \
    --model dinov2 \
    --device cuda \
    --one \
    --top_n 5 \
    --no_popup

# Display dashboard inline
from IPython.display import Image, display
dashboard_file = "./outputs_colab/reid_trajectory_result.jpg"
if os.path.exists(dashboard_file):
    print("\n🖼️ Rendered Trajectory Dashboard:")
    display(Image(dashboard_file, width=1200))
else:
    print("Dashboard image not found.")

## 🎯 Step 8: Search by Specific Vehicle Crop / Plate
Specify any vehicle crop path from `outputs_colab/crops/vehicles/` to trace its route across the other cameras.

In [ ]:
import glob
crops = sorted(glob.glob("./outputs_colab/crops/vehicles/*.jpg"))

if crops:
    # Select query crop (change index or filename as desired)
    selected_query = crops[0]
    print(f"Selected Query Image: {selected_query}")
    
    !python pipeline/test_embeddings.py \
        --output_dir ./outputs_colab \
        --model dinov2 \
        --device cuda \
        --image {selected_query} \
        --top_n 5 \
        --no_popup
        
    display(Image("./outputs_colab/reid_trajectory_result.jpg", width=1200))
else:
    print("No crop images found in ./outputs_colab/crops/vehicles/")

## 📈 Step 9: Re-ID Benchmark Evaluation
Evaluates Top-1 accuracy, Top-5 accuracy, and Mean Reciprocal Rank (MRR) across all multi-frame sample crops.

In [ ]:
!python pipeline/test_embeddings.py \
    --output_dir ./outputs_colab \
    --model dinov2 \
    --device cuda \
    --top_k 5 \
    --batch_size 64

## 💾 Step 10: Package, Export, and Download Results
Compresses the entire output directory (crops, embeddings, annotated videos, CSV logs, trajectory dashboards) into `outputs_colab.zip` and triggers automatic download to your local machine.

In [ ]:
import os
print("📦 Compressing outputs into outputs_colab.zip (crops, embeddings, logs, videos)...")
!zip -r -q outputs_colab.zip ./outputs_colab

zip_size = os.path.getsize("outputs_colab.zip") / (1024 * 1024)
print(f"✅ Archive created: outputs_colab.zip ({zip_size:.1f} MB)")

# Trigger browser download
from google.colab import files
print("⬇️ Starting download to your computer...")
files.download("outputs_colab.zip")

# Or save to Google Drive if mounted:
# !cp outputs_colab.zip /content/drive/MyDrive/SIH_Outputs_Colab.zip